# Legal Answer Server Quick Demo

This notebook only demos the legal answer server tool.
Run the code cells, then type your questions in the input box.
Type `quit` (or `q`/`exit`) to stop.

In [8]:
from pathlib import Path
import sys
from typing import Dict

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from legal_answer_server import answer_legal_question, answer_service_healthcheck

In [9]:
class LegalAnswerAgent:
    def __init__(self, top_k: int = 4, include_graph: bool = True, use_cache: bool = True):
        self.top_k = top_k
        self.include_graph = include_graph
        self.use_cache = use_cache

    def answer(self, question: str) -> Dict[str, str]:
        result = answer_legal_question(
            question=question,
            top_k=self.top_k,
            include_graph=self.include_graph,
            use_cache=self.use_cache,
        )

        if not result.get("success"):
            return {
                "final_answer": f"Tool failed: {result.get('error', 'unknown error')}",
                "tool_called": "answer_legal_question",
            }

        return {
            "final_answer": result.get("answer", "(empty answer)"),
            "tool_called": "answer_legal_question",
        }


health = answer_service_healthcheck()
print("Health success:", health.get("success"))
agent = LegalAnswerAgent()

2026-04-13 11:05:29,725 - LegalAnswerServer - INFO - Tool answer_service_healthcheck invoked
2026-04-13 11:05:31,878 - LegalAnswerServer - INFO - Tool answer_service_healthcheck completed success=False


Health success: False


In [ ]:
print("Type your legal question and press Enter.")
print("Type 'quit' (or 'q'/'exit') to finish.\n")

while True:
    question = input("Question: ").strip()

    if question.lower() in {"quit", "q", "exit"}:
        print("Session ended.")
        break

    if not question:
        print("Please enter a question, or type quit to stop.\n")
        continue

    response = agent.answer(question)
    print("Tool:", response["tool_called"])

    print("Final answer:")
    print(response["final_answer"])
    print()

Type your legal question and press Enter.
Type 'quit' (or 'q'/'exit') to finish.



## Usage tips
- Ask one legal question at a time for better retrieval quality.
- Type quit when you are done.
- If health check shows failure, confirm your env/config for Milvus, Neo4j, and provider keys.